# Colab Simulation: Gamma +.01 (Oganisian et al. 2019) on DGP B: Gamma Hurdle
- **Model**: Gamma +.01 (Oganisian et al. 2019) (`Gamma+.01`) — benchmark, shipped in `zicbcf`
- **DGP**: DGP B: Gamma Hurdle
- **Sample Size (N)**: 500
- **Zero-Inflation Intercept (c_shift)**: 1.0
- **MCMC**: NBURN = 1000, NSIM = 1000
- **Simulations**: 100 seeds
- **Output**: CSV with CATE metrics (Hurdle columns are NA: no hurdle component); `Status` flags any failed seed.


In [ ]:
# Install the zicbcf package from GitHub (ships BCF, ZIC-BCF, and the
# EDP+ZI / DPglm benchmark models with all their dependencies).
install.packages("remotes", repos="https://cloud.r-project.org/")
if (!require("devtools")) {
  install.packages("devtools", repos="https://cloud.r-project.org/")
}
devtools::install_github("hugogobato/zicbcf")
library(zicbcf)

In [ ]:
N_SIM   <- 100L
N       <- 500L
P       <- 5L
NBURN   <- 1000L
NSIM    <- 1000L
NTHIN   <- 1L

OUT_CSV <- "results_gamma_plus01_dgp_b_N500.csv"
if (file.exists(OUT_CSV)) file.remove(OUT_CSV)

In [ ]:
generate_dgp_b <- function(n, p, seed, c_shift = 0.2) {
  set.seed(seed * 1000 + 42)
  X <- matrix(rnorm(n * p), n, p)
  colnames(X) <- paste0("X", 1:p)

  pi_x <- pnorm(-0.5 + 0.4 * X[, 1] + 0.3 * X[, 2]^2)
  Z    <- rbinom(n, 1, pi_x)

  # Hurdle (similar treatment effect to A, halved)
  p_hurdle_0   <- pnorm(c_shift + 0.5 * X[, 1] - 0.3 * X[, 3])
  p_hurdle_1   <- pnorm(c_shift + 0.5 * X[, 1] - 0.3 * X[, 3] + 0.2 + 0.1 * X[, 1])
  p_hurdle_obs <- ifelse(Z == 1, p_hurdle_1, p_hurdle_0)
  I <- rbinom(n, 1, p_hurdle_obs)

  # Continuous intensity Gamma (right-skewed, zero-inflated, different from A & C)
  log_mu_c_0 <- 1.5 + 0.8 * X[, 2] + 0.4 * X[, 4]
  log_mu_c_1 <- 1.5 + 0.8 * X[, 2] + 0.4 * X[, 4] + 0.25 - 0.15 * X[, 2]

  mu_c_0 <- exp(log_mu_c_0)
  mu_c_1 <- exp(log_mu_c_1)

  alpha <- 2.0 # shape parameter for right-skewness
  scale_0 <- mu_c_0 / alpha
  scale_1 <- mu_c_1 / alpha

  y_pos_0 <- rgamma(n, shape = alpha, scale = scale_0)
  y_pos_1 <- rgamma(n, shape = alpha, scale = scale_1)
  y_pos_obs <- ifelse(Z == 1, y_pos_1, y_pos_0)
  Y <- I * y_pos_obs

  # True potential outcomes & CATE
  true_mu0  <- p_hurdle_0 * mu_c_0
  true_mu1  <- p_hurdle_1 * mu_c_1
  true_cate <- true_mu1 - true_mu0

  true_hurdle_cate <- p_hurdle_1 - p_hurdle_0

  list(y = Y, z = Z, x = X, pihat = pi_x, true_cate = true_cate, true_ate = mean(true_cate),
       true_hurdle_cate = true_hurdle_cate, true_hurdle_ate = mean(true_hurdle_cate))
}
calc_cate_metrics <- function(cate_draws, true_c, ate_draws) {
  cate_est <- colMeans(cate_draws)
  cate_ci  <- apply(cate_draws, 2, quantile, probs = c(0.025, 0.975))

  rmse <- sqrt(mean((cate_est - true_c)^2))
  bias <- mean(cate_est - true_c)
  coverage <- mean(true_c >= cate_ci[1, ] & true_c <= cate_ci[2, ])
  correlation <- cor(cate_est, true_c)
  if (is.na(correlation)) correlation <- 0.0
  ci_length <- mean(cate_ci[2, ] - cate_ci[1, ])
  est_ate_mean <- mean(ate_draws)

  list(rmse=rmse, bias=bias, coverage=coverage, correlation=correlation, ci_length=ci_length, est_ate_mean=est_ate_mean)
}

In [ ]:
cat("=== Starting Gamma +.01 (Oganisian et al. 2019) on DGP B: Gamma Hurdle ===\n")
for (s in 1:N_SIM) {
  cat(sprintf("[Seed %d/%d] Generating and fitting...\n", s, N_SIM))
  d <- generate_dgp_b(N, P, seed = s, c_shift = 1.0)

  fit <- tryCatch(
    gamma_plus01(y = d$y, z = d$z, x = d$x,
                      nburn = NBURN, nsim = NSIM, nthin = NTHIN),
    error = function(e) { message(sprintf("  seed %d FAILED: %s", s, conditionMessage(e))); NULL }
  )

  if (is.null(fit)) {
    # Record the failed seed transparently.
    df_res <- data.frame(
      DGP = "DGP B: Gamma Hurdle", Seed = s, Status = "FAILED",
      True_ATE = d$true_ate, True_Hurdle_ATE = d$true_hurdle_ate,
      GammaP01_CATE_RMSE = NA,
      GammaP01_CATE_Abs_Bias = NA,
      GammaP01_CATE_Coverage = NA,
      GammaP01_CATE_Correlation = NA,
      GammaP01_CATE_CI_Length = NA,
      GammaP01_Est_ATE = NA,
      GammaP01_Hurdle_RMSE = NA,
      GammaP01_Hurdle_Abs_Bias = NA,
      GammaP01_Hurdle_Coverage = NA,
      GammaP01_Hurdle_Correlation = NA,
      GammaP01_Hurdle_CI_Length = NA,
      GammaP01_Est_Hurdle_ATE = NA,
      stringsAsFactors = FALSE)
  } else {
    m  <- calc_cate_metrics(fit$cate, d$true_cate, fit$ate)
    df_res <- data.frame(
      DGP = "DGP B: Gamma Hurdle", Seed = s, Status = "OK",
      True_ATE = d$true_ate, True_Hurdle_ATE = d$true_hurdle_ate,
      GammaP01_CATE_RMSE = m$rmse,
      GammaP01_CATE_Abs_Bias = abs(m$bias),
      GammaP01_CATE_Coverage = m$coverage,
      GammaP01_CATE_Correlation = m$correlation,
      GammaP01_CATE_CI_Length = m$ci_length,
      GammaP01_Est_ATE = m$est_ate_mean,
      GammaP01_Hurdle_RMSE = NA,
      GammaP01_Hurdle_Abs_Bias = NA,
      GammaP01_Hurdle_Coverage = NA,
      GammaP01_Hurdle_Correlation = NA,
      GammaP01_Hurdle_CI_Length = NA,
      GammaP01_Est_Hurdle_ATE = NA,
      stringsAsFactors = FALSE)
  }

  write.table(df_res, OUT_CSV, sep=",", row.names=FALSE,
              col.names=!file.exists(OUT_CSV), append=TRUE)
  gc(verbose=FALSE)
}
cat("\n=== Finished Gamma +.01 (Oganisian et al. 2019) Run ===\n")